In [2]:
import os
import time
from unstract.llmwhisperer import LLMWhispererClientV2

from dotenv import load_dotenv
load_dotenv()

client = LLMWhispererClientV2(base_url='https://llmwhisperer-api.eu-west.unstract.com/api/v2',
                              api_key=os.getenv('API_KEY'))

pdf_file_path = r'C:\Users\ALTHAAF\Documents\Code\Finance\CureForFinancialInsomnia\input\AbansElec2.pdf'
pdf_name = os.path.splitext(os.path.basename(pdf_file_path))[0]
result = client.whisper(file_path=pdf_file_path)

while True:
    status = client.whisper_status(whisper_hash=result['whisper_hash'])
    if status['status'] == 'processed':
        resultx = client.whisper_retrieve(whisper_hash=result['whisper_hash'])
        break

    time.sleep(5)

extracted_table = resultx['extraction']['result_text']

2026-07-21 17:03:59,892 - unstract.llmwhisperer.client_v2 - DEBUG - logging_level set to DEBUG
2026-07-21 17:03:59,895 - unstract.llmwhisperer.client_v2 - DEBUG - base_url set to https://llmwhisperer-api.eu-west.unstract.com/api/v2
2026-07-21 17:03:59,897 - unstract.llmwhisperer.client_v2 - DEBUG - whisper called
2026-07-21 17:03:59,899 - unstract.llmwhisperer.client_v2 - DEBUG - api_url: https://llmwhisperer-api.eu-west.unstract.com/api/v2/whisper
2026-07-21 17:03:59,900 - unstract.llmwhisperer.client_v2 - DEBUG - params: {'mode': 'form', 'output_mode': 'layout_preserving', 'page_seperator': '<<<', 'pages_to_extract': '', 'median_filter_size': 0, 'gaussian_blur_radius': 0, 'line_splitter_tolerance': 0.4, 'horizontal_stretch_factor': 1.0, 'mark_vertical_lines': False, 'mark_horizontal_lines': False, 'line_spitter_strategy': 'left-priority', 'add_line_nos': False, 'include_line_confidence': False, 'lang': 'eng', 'tag': 'default', 'filename': '', 'webhook_metadata': '', 'use_webhook': ''

In [3]:
print(extracted_table)



         ABANS ELECTRICALS PLC                                                                                                                                                                                                                       ABANS ELECTRICALS PLC 
         Annual Report 2024/25                                                                                                                                                                                                                            Annual Report 2024/25 

 TEN YEAR SUMMARY 

31 March                                                                 2025           2024          2023          2022                                        2021         2020           2019           2018          2017         2016 

Operating Results In LKR '000 
Revenue                                                             6,707,214      6,123,806      5,246,866    4,329,171                                   3,129,718    

In [4]:
import re
import os
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML

# 1. Helper function to parse individual numeric tokens
def parse_number_from_token(token):
    t = token.replace('−', '-').replace('–', '-').replace('—', '-')
    if re.match(r'^\d{4}[-/]\d{2,4}$', t):
        return None
    is_negative = False
    cleaned_t = t.strip()
    if cleaned_t.startswith('(') and cleaned_t.endswith(')'):
        is_negative = True
    elif cleaned_t.startswith('-'):
        is_negative = True
    cleaned = t.replace(',', '').replace('(', '').replace(')', '').replace('%', '').replace('$', '').replace('€', '').replace('£', '').replace('-', '').strip()
    if cleaned.replace('.', '', 1).isdigit():
        val = float(cleaned)
        return -val if is_negative else val
    elif cleaned == '' or cleaned.lower() in ['n/a', 'na', 'nil']:
        return 0.0
    return None

# 2. Candidate extraction helper function
def extract_candidates_from_text(raw_text):
    lines = raw_text.strip().split('\n')
    
    year_pattern = re.compile(r'\b\d{4}(?:/\d{2,4}|-\d{2,4})?\b')
    all_year_tokens = year_pattern.findall(raw_text)
    
    seen_years = set()
    candidate_years = []
    for yr in all_year_tokens:
        if yr not in seen_years:
            seen_years.add(yr)
            candidate_years.append(yr)
            
    candidate_rows = []
    for i, line in enumerate(lines):
        tokens = line.strip().split()
        if not tokens:
            continue
        
        nums = []
        text_parts = []
        for tok in tokens:
            val = parse_number_from_token(tok)
            if val is not None:
                nums.append(val)
            else:
                text_parts.append(tok)
                
        if nums:
            label = " ".join(text_parts).strip()
            nums_preview = ", ".join(str(n) for n in nums[:3])
            if len(nums) > 3:
                nums_preview += "..."
            
            display_lbl = label if label else "[No Label]"
            display_name = f"Row {i+1}: {display_lbl} | [{nums_preview}]"
            candidate_rows.append((display_name, i, label, tokens))
            
    return candidate_years, candidate_rows

candidate_years, candidate_rows = extract_candidates_from_text(extracted_table)

# 3. Column Index Guide
guide_html = "<h3 style='color:#202124; margin-bottom: 5px;'>Column Index Guide (First 4 rows with numbers)</h3>"
guide_html += "<table border='1' style='border-collapse:collapse; cellpadding:5px; font-family:monospace;'>"
guide_html += "<tr style='background-color:#f1f3f4;'><th>Row Source</th>"
max_cols_preview = 8
for c in range(max_cols_preview):
    guide_html += f"<th>Col {c}</th>"
guide_html += "</tr>"

for row in candidate_rows[:4]:
    disp_name, line_idx, lbl, tokens = row
    num_tokens = [t for t in tokens if parse_number_from_token(t) is not None]
    guide_html += f"<tr><td><b>Row {line_idx+1}:</b> {lbl if lbl else '[No Label]'}</td>"
    for c in range(max_cols_preview):
        val = num_tokens[c] if c < len(num_tokens) else ""
        guide_html += f"<td>{val}</td>"
    guide_html += "</tr>"
guide_html += "</table>"
guide_html += "<p style='font-size:12px; color:#5f6368; margin-top:5px;'>Use this guide to identify column indices for absolute metrics.</p>"

target_keys = {
    "revenue": "Revenue",
    "pbt": "Profit/(Loss) before taxation",
    "total_equity": "Total equity",
    "total_borrowings": "Total borrowings",
    "current_assets": "Current assets",
    "total_assets": "Total assets employed",
    "current_ratio": "Current ratio (No. of times)",
    "quick_ratio": "Quick asset ratio",
    "gearing_ratio": "Gearing ratio (%)"
}

default_years = ", ".join(candidate_years[:6])
years_input = widgets.Text(
    value=default_years,
    description='Target Years (comma separated):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='60%')
)

col_indices_input = widgets.Text(
    value="",
    placeholder='e.g., 0, 2, 4 (Leave blank for consecutive 0, 1, 2...)',
    description='Column Indices to Extract (comma separated):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='60%')
)

# 4. Build Interactive Dynamic Controls per metric
options = [("-- Select Row --", -1)] + [(row[0], row[1]) for row in candidate_rows]
metric_widgets = {}

def make_add_term_handler(terms_list, vbox, row_options):
    def on_add_clicked(b):
        is_first = len(terms_list) == 0
        dd_w = widgets.Dropdown(options=row_options, value=-1, layout=widgets.Layout(width='70%'))
        
        if is_first:
            op_w = widgets.Label(value='+', layout=widgets.Layout(width='50px'))
            row_box = widgets.HBox([widgets.Label('Row 1:', layout=widgets.Layout(width='60px')), dd_w])
            term_dict = {'op': op_w, 'dropdown': dd_w, 'box': row_box}
        else:
            op_w = widgets.Dropdown(options=['+', '-'], value='+', layout=widgets.Layout(width='60px'))
            btn_del = widgets.Button(description='[X]', layout=widgets.Layout(width='45px'), button_style='danger')
            row_box = widgets.HBox([op_w, dd_w, btn_del])
            term_dict = {'op': op_w, 'dropdown': dd_w, 'box': row_box}
            
            def make_del_handler(td):
                def on_del_clicked(b_del):
                    if td in terms_list:
                        terms_list.remove(td)
                        vbox.children = tuple(t['box'] for t in terms_list)
                return on_del_clicked

            btn_del.on_click(make_del_handler(term_dict))
            
        terms_list.append(term_dict)
        vbox.children = tuple(t['box'] for t in terms_list)
    return on_add_clicked

for json_key, default_lbl in target_keys.items():
    selected_line_idx = -1
    for row in candidate_rows:
        disp_name, line_idx, lbl, tokens = row
        if lbl and (default_lbl.lower() in lbl.lower() or lbl.lower() in default_lbl.lower()):
            selected_line_idx = line_idx
            break
            
    if selected_line_idx == -1:
        if json_key == "revenue":
            for row in candidate_rows:
                disp_name, line_idx, lbl, tokens = row
                if lbl and ("revenue" in lbl.lower() or "turnover" in lbl.lower() or "income" in lbl.lower()) and "reserve" not in lbl.lower():
                    selected_line_idx = line_idx
                    break
        elif json_key == "pbt":
            for row in candidate_rows:
                disp_name, line_idx, lbl, tokens = row
                if lbl and ("before taxation" in lbl.lower() or "before tax" in lbl.lower() or "pbt" in lbl.lower()):
                    selected_line_idx = line_idx
                    break
        elif json_key == "total_assets":
            for row in candidate_rows:
                disp_name, line_idx, lbl, tokens = row
                if lbl and ("total assets" in lbl.lower()):
                    selected_line_idx = line_idx
                    break

    mode_toggle = widgets.ToggleButtons(
        options=['Direct Row', 'Dynamic Formula (Row Calculation)'],
        value='Direct Row',
        description='',
        button_style='',
        style={'button_width': '220px'}
    )
    
    dropdown_direct = widgets.Dropdown(
        options=options,
        value=selected_line_idx if any(val == selected_line_idx for _, val in options) else -1,
        description='Direct Row:',
        style={'description_width': '100px'},
        layout=widgets.Layout(width='75%')
    )
    
    formula_terms = []
    terms_vbox = widgets.VBox([])
    add_term_fn = make_add_term_handler(formula_terms, terms_vbox, options)
    
    # Pre-populate Term 1 & Term 2
    add_term_fn(None)
    add_term_fn(None)
    
    btn_add_term = widgets.Button(
        description='+ Add Row Term',
        button_style='info',
        icon='plus',
        layout=widgets.Layout(margin='5px 0px 5px 60px', width='160px')
    )
    btn_add_term.on_click(add_term_fn)
    
    formula_container = widgets.VBox([terms_vbox, btn_add_term], layout=widgets.Layout(display='none'))
    
    def make_toggle_handler(dd_dir, f_cont):
        def on_mode_change(change):
            if change['new'] == 'Direct Row':
                dd_dir.layout.display = ''
                f_cont.layout.display = 'none'
            else:
                dd_dir.layout.display = 'none'
                f_cont.layout.display = ''
        return on_mode_change

    mode_toggle.observe(make_toggle_handler(dropdown_direct, formula_container), names='value')
    
    card = widgets.VBox([
        widgets.HTML(f"<b>{json_key}</b> (Default: '{default_lbl}')"),
        mode_toggle,
        dropdown_direct,
        formula_container
    ], layout=widgets.Layout(border='1px solid #dadce0', padding='10px', margin='5px 0px'))
    
    metric_widgets[json_key] = {
        'card': card,
        'mode': mode_toggle,
        'direct': dropdown_direct,
        'terms': formula_terms
    }

btn_extract = widgets.Button(
    description='Run Extraction',
    button_style='success',
    tooltip='Click to parse metrics and run financial analysis',
    icon='check',
    layout=widgets.Layout(margin='15px 0px 0px 0px')
)

output_area = widgets.Output()

def get_row_numbers(line_idx):
    lines = extracted_table.strip().split('\n')
    if line_idx < 0 or line_idx >= len(lines):
        return []
    line = lines[line_idx]
    tokens = line.strip().split()
    clean_nums = []
    for tok in tokens:
        val = parse_number_from_token(tok)
        if val is not None:
            clean_nums.append(val)
    return clean_nums

def on_btn_clicked(b):
    with output_area:
        output_area.clear_output()
        
        target_yrs = [yr.strip() for yr in years_input.value.split(",") if yr.strip()]
        if not target_yrs:
            print("❌ Please specify at least one target year.")
            return
            
        try:
            col_indices = [int(idx.strip()) for idx in col_indices_input.value.split(",") if idx.strip()]
        except Exception:
            print("❌ Invalid Column Indices format.")
            return
            
        if not col_indices:
            col_indices = list(range(len(target_yrs)))
            
        data_store = {year: {} for year in target_yrs}
        
        for json_key, w_dict in metric_widgets.items():
            mode = w_dict['mode'].value
            if mode == 'Direct Row':
                line_idx = w_dict['direct'].value
                if line_idx != -1:
                    clean_numbers = get_row_numbers(line_idx)
                    for year_idx, year in enumerate(target_yrs):
                        if year_idx < len(col_indices):
                            t_col = col_indices[year_idx]
                            if t_col < len(clean_numbers):
                                data_store[year][json_key] = clean_numbers[t_col]
            else:
                terms = w_dict['terms']
                for year_idx, year in enumerate(target_yrs):
                    if year_idx < len(col_indices):
                        t_col = col_indices[year_idx]
                        tot_val = 0.0
                        has_valid_term = False
                        for idx_t, term in enumerate(terms):
                            line_idx = term['dropdown'].value
                            if line_idx != -1:
                                nums = get_row_numbers(line_idx)
                                val = nums[t_col] if t_col < len(nums) else 0.0
                                op = '+' if idx_t == 0 else term['op'].value
                                if op == '+':
                                    tot_val += val
                                else:
                                    tot_val -= val
                                has_valid_term = True
                        if has_valid_term:
                            data_store[year][json_key] = tot_val
                            
        for year in data_store:
            for k in target_keys.keys():
                if k not in data_store[year]:
                    data_store[year][k] = 0.0
                    
        global df, analysis_df, original_df, analysis_results_df, trimmed_analysis_df
        df = pd.DataFrame(data_store).T
        df.index.name = 'Year'
        df = df.reset_index()
        
        print("✅ Raw Data Extracted (with Multi-Row Derived Formulas):")
        display(df)
        
        try:
            analysis_df = df.copy()
            required_cols = ['revenue', 'pbt', 'total_equity', 'total_borrowings', 'current_assets', 'total_assets', 'current_ratio', 'quick_ratio', 'gearing_ratio']
            for col in required_cols:
                if col not in analysis_df.columns:
                    analysis_df[col] = 0.0
                else:
                    analysis_df[col] = pd.to_numeric(analysis_df[col]).fillna(0.0)
                    
            base_year_idx = max(0, len(analysis_df) - 1)
            base_year_row = analysis_df.loc[base_year_idx]
            core_metrics = ['revenue', 'pbt', 'total_equity', 'total_borrowings', 'current_assets', 'total_assets']
            
            safe_current_ratio = analysis_df['current_ratio'].replace(0, 1.0)
            current_liabs = analysis_df['current_assets'] / safe_current_ratio
            analysis_df['net_working_capital_abs'] = analysis_df['current_assets'] - current_liabs
            
            for metric in core_metrics:
                analysis_df[f'{metric}_horizontal_growth_%'] = analysis_df[metric].pct_change(-1) * 100
                base_val = base_year_row[metric] if base_year_row[metric] != 0 else 1.0
                analysis_df[f'{metric}_trend_index'] = (analysis_df[metric] / base_val) * 100
                
            safe_total_assets = analysis_df['total_assets'].replace(0, 1.0)
            safe_revenue = analysis_df['revenue'].replace(0, 1.0)
            analysis_df['current_assets_vertical_%'] = (analysis_df['current_assets'] / safe_total_assets) * 100
            analysis_df['total_equity_vertical_%'] = (analysis_df['total_equity'] / safe_total_assets) * 100
            analysis_df['borrowings_vertical_%'] = (analysis_df['total_borrowings'] / safe_total_assets) * 100
            analysis_df['pbt_vertical_margin_%'] = (analysis_df['pbt'] / safe_revenue) * 100
            
            analysis_df['net_profit_margin_%'] = (analysis_df['pbt'] / safe_revenue) * 100
            analysis_df['return_on_assets_roa_%'] = (analysis_df['pbt'] / safe_total_assets) * 100
            safe_total_equity = analysis_df['total_equity'].replace(0, 1.0)
            analysis_df['return_on_equity_roe_%'] = (analysis_df['pbt'] / safe_total_equity) * 100
            analysis_df['working_capital_to_assets_%'] = (analysis_df['net_working_capital_abs'] / safe_total_assets) * 100
            analysis_df['debt_to_equity_ratio'] = analysis_df['total_borrowings'] / safe_total_equity
            analysis_df['equity_to_total_assets_%'] = (analysis_df['total_equity'] / safe_total_assets) * 100
            
            analysis_df['Z_X1'] = analysis_df['net_working_capital_abs'] / safe_total_assets
            analysis_df['Z_X2'] = analysis_df['total_equity'] / safe_total_assets
            analysis_df['Z_X3'] = analysis_df['pbt'] / safe_total_assets
            safe_total_borrowings = analysis_df['total_borrowings'].replace(0, 1.0)
            analysis_df['Z_X4'] = analysis_df['total_equity'] / safe_total_borrowings
            analysis_df['Z_X5'] = analysis_df['revenue'] / safe_total_assets
            
            analysis_df['altman_z_score'] = (1.2 * analysis_df['Z_X1']) + (1.4 * analysis_df['Z_X2']) + \
                                           (3.3 * analysis_df['Z_X3']) + (0.6 * analysis_df['Z_X4']) + \
                                           (0.999 * analysis_df['Z_X5'])
                                           
            def categorize_z_zone(z):
                if z > 2.90: return "Safe Zone"
                elif z < 1.23: return "Distress Zone"
                return "Grey Zone"
                
            analysis_df['altman_risk_classification'] = analysis_df['altman_z_score'].apply(categorize_z_zone)
            analysis_df = analysis_df.round(2)
            
            trim_len = max(1, len(target_yrs) - 1)
            trimmed_analysis_df = analysis_df.iloc[:trim_len].copy()
            
            original_cols = ['Year', 'revenue', 'pbt', 'total_equity', 'total_borrowings', 
                             'current_assets', 'total_assets', 'gearing_ratio', 'current_ratio', 'quick_ratio']
            for col in original_cols:
                if col not in analysis_df.columns:
                    analysis_df[col] = 0.0
            original_df = analysis_df[original_cols].copy()
            
            analysis_cols = [col for col in analysis_df.columns if col not in original_cols or col == 'Year']
            analysis_results_df = analysis_df[analysis_cols].iloc[:trim_len].copy()
            
            print("📈 Financial Analysis Calculated successfully!")
            print("\n--- Original Metrics ---")
            display(original_df)
            print("\n--- Analysis Results ---")
            display(analysis_results_df)
            
            excel_name = f"{pdf_name}_derived_analysis.xlsx"
            folder_path = "output"
            if not os.path.exists(folder_path):
                os.makedirs(folder_path)
            excel_path = os.path.join(folder_path, excel_name)
            
            with pd.ExcelWriter(excel_path) as writer:
                original_df.to_excel(writer, sheet_name='Original Metrics', index=False)
                analysis_results_df.to_excel(writer, sheet_name='Analysis Results', index=False)
                
            print(f"\n💾 Excel file '{excel_name}' created successfully in '{folder_path}' folder.")
            
        except Exception as ex:
            print(f"❌ Error during calculation or Excel saving: {ex}")
            import traceback
            traceback.print_exc()

btn_extract.on_click(on_btn_clicked)

display(HTML("<h2 style='color:#1a73e8; margin-top:0px;'>Interactive Derived Financial Parser Mapping (v2 Multi-Row)</h2>"))
display(HTML(guide_html))
display(years_input)
display(col_indices_input)

cards_list = [mw['card'] for mw in metric_widgets.values()]
display(widgets.VBox(cards_list))
display(btn_extract)
display(output_area)


Row Source,Col 0,Col 1,Col 2,Col 3,Col 4,Col 5,Col 6,Col 7
Row 6: March,31,2025,2024,2023,2022,2021,2020,2019
Row 9: Revenue,"6,707,214","6,123,806","5,246,866","4,329,171","3,129,718","2,986,136","2,870,558","3,492,566"
Row 10: Gross Profit,"1,605,269","1,376,778","876,335","548,665","473,355","441,348","397,554","392,845"
Row 11: EBIT,"770,354","772,748","482,011","216,058","232,733","202,539","144,017","154,356"


Text(value='2024/25, 2025, 2024, 2023, 2022, 2021', description='Target Years (comma separated):', layout=Layo…

Text(value='', description='Column Indices to Extract (comma separated):', layout=Layout(width='60%'), placeho…

Button(button_style='success', description='Run Extraction', icon='check', layout=Layout(margin='15px 0px 0px …

Output()

In [ ]:
df

In [ ]:
original_df

In [ ]:
analysis_results_df